# Pre-trained Encoder + JacobianODE — Lorenz (Single Run)

This notebook loads a **pre-trained sequence encoder** (from the Encoder-Only pipeline)
and trains a **JacobianODE** model in the learned latent space.

**Pipeline:**
1. Load pre-trained encoder + decoder from a W&B checkpoint
2. Generate/load the same data the encoder was trained on
3. Build `LitLatentJacobianODE` using the encoder (frozen or unfrozen)
4. Train: observations → encoder → latent → JacobianODE → decoder → obs-space loss
5. Evaluate: prediction quality, Lyapunov exponents, latent structure

**Key features:**
- **Frozen encoder**: only the Jacobian MLP trains (fast, stable)
- **Unfrozen encoder**: encoder fine-tunes jointly with JacobianODE (can improve latent geometry)
- Losses in **both** latent space (JacODE prediction, loop closure) and observation space (trajectory prediction, reconstruction)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.pyplot as plt
import numpy as np
from omegaconf import OmegaConf
import os
import torch
import wandb

from JacobianODE.jacobians import (
    load_config,
    initialize_config,
    seed_everything,
    make_trajectories,
    postprocess_data,
    create_dataloaders,
    train_model,
)
from JacobianODE.encoder_only.pretrained import load_pretrained_encoder
from JacobianODE.models.latent_jacobian import LitLatentJacobianODE
from JacobianODE.fnn import compute_variances, compute_s_dim
from hydra.utils import instantiate

torch.set_float32_matmul_precision('high')

In [3]:
# ----------------------------------------------------------------
# Paths and W&B settings
# ----------------------------------------------------------------
SAVE_DIR = "/orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/pretrained_jac_runs"
ENCODER_SAVE_DIR = "/orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/encoders"
WANDB_ENTITY = "JacobianODE"
# WANDB_PROJECT = None  # Auto-generated below from data class
WANDB_PROJECT = "Lorenz__PretrainedEncoderJacODE"

# ----------------------------------------------------------------
# Pre-trained encoder source (EDIT THESE)
# ----------------------------------------------------------------
# ENCODER_PROJECT = "JacobianODE/<YOUR_ENCODER_PROJECT>"  # e.g. "JacobianODE/Lorenz__EncoderOnly"
# ENCODER_RUN_ID = "<YOUR_RUN_ID>"                        # e.g. "abc12345"

ENCODER_PROJECT = "LorenzPartial_SEQ250BURN50K1D43__Encoder"
ENCODER_RUN_ID = "iu8cq3qx"

## 1. Load Pre-trained Encoder

Set your encoder's W&B project and run ID below. The encoder, its same-state decoder,
and training config are loaded from the checkpoint.

In [4]:
# ----------------------------------------------------------------
# Encoder freeze/unfreeze
# ----------------------------------------------------------------
# True  = only the Jacobian MLP trains (faster, simpler)
# False = encoder fine-tunes jointly (needs obs-space losses to stay grounded)
FREEZE_ENCODER = True

In [5]:
adapter, encoder_cfg, encoder_run = load_pretrained_encoder(
    project=ENCODER_PROJECT,
    run_id=ENCODER_RUN_ID,
    save_dir=ENCODER_SAVE_DIR,
    freeze=FREEZE_ENCODER,
    verbose=True,
)

N_LATENT = adapter.n_latent
print(f"Encoder type:    {type(adapter.encoder).__name__}")
print(f"n_latent:        {N_LATENT}")
print(f"context_margin:  {adapter.context_margin}")
print(f"Frozen:          {FREEZE_ENCODER}")
print(f"Encoder params:  {sum(p.numel() for p in adapter.parameters()):,}")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/eisenaj/.netrc.


No checkpoint for best epoch 23; using epoch=24-step=1250.ckpt instead.


Encoder type:    SSMSequenceEncoder
n_latent:        10
context_margin:  0
Frozen:          True
Encoder params:  773,045


## 2. Data Hyperparameters

These are pulled from the encoder's training config to ensure the same data
generation and preprocessing pipeline.

In [6]:
# Pull data config from the encoder training run
NUM_ICS = int(encoder_cfg.data.trajectory_params.num_ics)
N_PERIODS = int(encoder_cfg.data.trajectory_params.n_periods)
PTS_PER_PERIOD = int(encoder_cfg.data.trajectory_params.pts_per_period)
SEQ_LENGTH = int(encoder_cfg.data.train_test_params.seq_length)
OBS_NOISE = float(encoder_cfg.data.postprocessing.obs_noise)

delay_params = encoder_cfg.data.train_test_params.delay_embedding_params
OBSERVED_INDICES = list(delay_params.observed_indices) if delay_params.observed_indices != "all" else "all"
N_DELAYS = int(delay_params.get("n_delays", 1))
DELAY_SPACING = int(delay_params.get("delay_spacing", 1))

print(f"Data config (from encoder):")
print(f"  NUM_ICS={NUM_ICS}, N_PERIODS={N_PERIODS}, PTS_PER_PERIOD={PTS_PER_PERIOD}")
print(f"  SEQ_LENGTH={SEQ_LENGTH}, OBS_NOISE={OBS_NOISE}")
print(f"  OBSERVED_INDICES={OBSERVED_INDICES}, N_DELAYS={N_DELAYS}")

Data config (from encoder):
  NUM_ICS=32, N_PERIODS=12, PTS_PER_PERIOD=100
  SEQ_LENGTH=100, OBS_NOISE=0.05
  OBSERVED_INDICES=[0], N_DELAYS=43


## 3. JacobianODE Hyperparameters

In [7]:
# ----------------------------------------------------------------
# JacobianODE integration
# ----------------------------------------------------------------
# PREDICTION_STEPS = 10
PREDICTION_STEPS = 30
TRAJ_INIT_STEPS = 15
INTERP_PTS = 4
INNER_N = 20

# ----------------------------------------------------------------
# Jacobian MLP architecture
# ----------------------------------------------------------------
JAC_HIDDEN_DIM = [256, 512, 512]
JAC_NUM_LAYERS = 3
JAC_ACTIVATION = 'silu'

# ----------------------------------------------------------------
# Loss weights
# ----------------------------------------------------------------
LOOP_CLOSURE_WEIGHT = 0.001
# LOOP_CLOSURE_WEIGHT = 0.0

# Obs-space losses:
#   - When frozen:   set reconstruction to 0 (it's constant)
#   - When unfrozen: keep > 0 to prevent encoder drift
RECONSTRUCTION_LOSS_WEIGHT = 0.0 if FREEZE_ENCODER else 1.0

# Latent-space losses:
# LATENT_PREDICTION_LOSS_WEIGHT = 0.0 if FREEZE_ENCODER else 1.0
LATENT_PREDICTION_LOSS_WEIGHT = 1.0
JAC_CONSISTENCY_WEIGHT = 0.0
FNN_WEIGHT = 0.0 if FREEZE_ENCODER else 0.001

# ----------------------------------------------------------------
# Training
# ----------------------------------------------------------------
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 200
# LIMIT_TRAIN_BATCHES = 500
LIMIT_TRAIN_BATCHES = 200
LIMIT_VAL_BATCHES = 50
ACCUMULATE_GRAD_BATCHES = 4
# EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_PATIENCE = 2
PERCENT_THRESH = 0.01


# ----------------------------------------------------------------
# Homoscedastic uncertainty weighting (Kendall et al. 2018)
# ----------------------------------------------------------------
LEARN_R2_WEIGHT = False
LEARN_LOOP_CLOSURE_WEIGHT = False
LEARN_FNN_WEIGHT = False
LEARN_JAC_CONS_WEIGHT = False
LEARN_JAC_NORM_WEIGHT = False
LOG_VAR_INIT = 'naive'

In [8]:
# ----------------------------------------------------------------
# Verify sequence length is sufficient for JacobianODE windows
# ----------------------------------------------------------------
# For sequence-based encoders (no time_window), the latent trajectory
# length is T' = SEQ_LENGTH - context_margin.
# JacobianODE needs: T' >= TRAJ_INIT_STEPS + PREDICTION_STEPS
T_latent = SEQ_LENGTH - adapter.context_margin
JAC_WINDOW = TRAJ_INIT_STEPS + PREDICTION_STEPS
assert T_latent >= JAC_WINDOW, (
    f"Latent trajectory length T'={T_latent} (SEQ_LENGTH={SEQ_LENGTH} - "
    f"context_margin={adapter.context_margin}) is too short for "
    f"JacobianODE window={JAC_WINDOW} (init={TRAJ_INIT_STEPS} + pred={PREDICTION_STEPS}). "
    f"Increase SEQ_LENGTH or reduce TRAJ_INIT_STEPS/PREDICTION_STEPS."
)
print(f"Latent trajectory length: T' = {T_latent}")
print(f"JacobianODE window:       {TRAJ_INIT_STEPS} + {PREDICTION_STEPS} = {JAC_WINDOW}")
print(f"Sub-windows per sample:   ~{(T_latent - JAC_WINDOW) // PREDICTION_STEPS + 1}")

Latent trajectory length: T' = 100
JacobianODE window:       15 + 30 = 45
Sub-windows per sample:   ~2


## 4. Build Config

We use `model=latent_ssm` as the base config template (to get the right config
shape with an `encoder` block), then override all relevant parameters.
The encoder from config is **not used** — we inject the pre-trained adapter at
model build time.

In [9]:
_hidden_dim_str = str(JAC_HIDDEN_DIM).replace(' ', '')
_obs_idx_str = str(list(OBSERVED_INDICES)).replace(' ', '') if OBSERVED_INDICES != "all" else "all"

# Propagate actual encoder architecture from the pretrained encoder config
# (encoder_cfg comes from the encoder W&B run, not the latent_ssm template)
_enc = encoder_cfg.model.encoder
_enc_d_model = int(_enc.get("d_model", 64))
_enc_d_state = int(_enc.get("d_state", 64))
_enc_n_layers = int(_enc.get("n_layers", 3))
_enc_ffn_expand = int(_enc.get("ffn_expand", 2))
_enc_r_min = float(_enc.get("r_min", 0.0))
_enc_r_max = float(_enc.get("r_max", 0.99))
_enc_dropout = float(_enc.get("dropout", 0.1))
_enc_use_pe = bool(_enc.get("use_positional_encoding", True))
_enc_pe_type = str(_enc.get("positional_encoding_type", "sinusoidal"))
_enc_dec_hidden = int(_enc.get("decoder_hidden", encoder_cfg.model.get("decoder_hidden_dim", 128)))
_enc_dec_layers = int(_enc.get("decoder_layers", encoder_cfg.model.get("decoder_n_layers", 2)))

overrides = [
    # --- Model: use latent_ssm as config template ---
    "model=latent_ssm",
    f"model.prediction_steps={PREDICTION_STEPS}",
    
    # --- Encoder: propagate actual pretrained encoder architecture ---
    f"model.encoder.n_latent={N_LATENT}",
    f"model.encoder.d_model={_enc_d_model}",
    f"model.encoder.d_state={_enc_d_state}",
    f"model.encoder.n_layers={_enc_n_layers}",
    f"model.encoder.ffn_expand={_enc_ffn_expand}",
    f"model.encoder.r_min={_enc_r_min}",
    f"model.encoder.r_max={_enc_r_max}",
    f"model.encoder.dropout={_enc_dropout}",
    f"model.encoder.use_positional_encoding={_enc_use_pe}",
    f"model.encoder.positional_encoding_type={_enc_pe_type}",
    f"model.encoder.decoder_hidden={_enc_dec_hidden}",
    f"model.encoder.decoder_layers={_enc_dec_layers}",
    f"model.encoder.context_margin={adapter.context_margin}",
    
    # --- Jacobian MLP ---
    f"model.params.hidden_dim={_hidden_dim_str}",
    f"model.params.num_layers={JAC_NUM_LAYERS}",
    f"model.params.activation={JAC_ACTIVATION}",
    
    # --- Data (must match encoder training) ---
    "data=dysts",
    "data.flow._target_=JacobianODE.dysts_sim.flows.Lorenz",
    f"data.trajectory_params.num_ics={NUM_ICS}",
    f"data.trajectory_params.n_periods={N_PERIODS}",
    f"data.trajectory_params.pts_per_period={PTS_PER_PERIOD}",
    f"data.postprocessing.obs_noise={OBS_NOISE}",
    f"data.train_test_params.seq_length={SEQ_LENGTH}",
    f"data.train_test_params.delay_embedding_params.observed_indices={_obs_idx_str}",
    f"data.train_test_params.delay_embedding_params.n_delays={N_DELAYS}",
    f"data.train_test_params.delay_embedding_params.delay_spacing={DELAY_SPACING}",
    
    # --- Training ---
    f"training.batch_size={BATCH_SIZE}",
    f"training.logger.save_dir={SAVE_DIR}",
    f"training.lightning.optimizer_kwargs.lr={LEARNING_RATE}",
    f"training.lightning.optimizer_kwargs.weight_decay={WEIGHT_DECAY}",
    
    # --- Loss weights ---
    f"training.lightning.loop_closure_weight={LOOP_CLOSURE_WEIGHT}",
    f"training.lightning.reconstruction_loss_weight={RECONSTRUCTION_LOSS_WEIGHT}",
    f"training.lightning.latent_prediction_loss_weight={LATENT_PREDICTION_LOSS_WEIGHT}",
    f"training.lightning.jac_consistency_weight={JAC_CONSISTENCY_WEIGHT}",
    f"training.lightning.fnn_weight={FNN_WEIGHT}",
    
    # --- Homoscedastic uncertainty weighting ---
    f"training.lightning.learn_r2_weight={LEARN_R2_WEIGHT}",
    f"training.lightning.learn_loop_closure_weight={LEARN_LOOP_CLOSURE_WEIGHT}",
    f"training.lightning.learn_fnn_weight={LEARN_FNN_WEIGHT}",
    f"training.lightning.learn_jac_cons_weight={LEARN_JAC_CONS_WEIGHT}",
    f"training.lightning.learn_jac_norm_weight={LEARN_JAC_NORM_WEIGHT}",
    f"training.lightning.log_var_init={LOG_VAR_INIT}",
    
    # --- Teacher forcing ---
    "training.lightning.alpha_teacher_forcing=1",
    "training.lightning.teacher_forcing_annealing=True",
    "training.lightning.gamma_teacher_forcing=0.999",
    "training.lightning.loop_closure_training=True",
    "training.lightning.trajectory_training=True",
    
    # --- JacobianODEint ---
    f"training.lightning.jacobianODEint_kwargs.traj_init_steps={TRAJ_INIT_STEPS}",
    f"training.lightning.jacobianODEint_kwargs.interp_pts={INTERP_PTS}",
    f"training.lightning.jacobianODEint_kwargs.inner_N={INNER_N}",
    "training.lightning.jacobianODEint_kwargs.inner_path=line",
    
    # --- Trainer params ---
    f"training.trainer_params.max_epochs={MAX_EPOCHS}",
    f"training.trainer_params.limit_train_batches={LIMIT_TRAIN_BATCHES}",
    f"training.trainer_params.limit_val_batches={LIMIT_VAL_BATCHES}",
    f"training.trainer_params.accumulate_grad_batches={ACCUMULATE_GRAD_BATCHES}",
    
    # --- Early stopping ---
    f"training.early_stopping.early_stopping_patience={EARLY_STOPPING_PATIENCE}",
    f"++training.early_stopping.percent_thresh={PERCENT_THRESH}",
]

cfg = load_config(overrides=overrides)
cfg = initialize_config(cfg)

print(f"Lightning target:  {cfg.training.lightning._target_}")
print(f"Jacobian MLP:      input_dim={cfg.model.params.input_dim}, output_dim={cfg.model.params.output_dim}")
print(f"Config n_latent:   {cfg.model.encoder.n_latent}")
print(f"Config d_model:    {cfg.model.encoder.d_model}")
print(f"Config ffn_expand: {cfg.model.encoder.ffn_expand}")


Lightning target:  JacobianODE.models.latent_jacobian.LitLatentJacobianODE
Jacobian MLP:      input_dim=10, output_dim=100
Config n_latent:   10
Config d_model:    128
Config ffn_expand: 4


## 5. Generate Data

Generate trajectories with the same config the encoder was trained on.

In [10]:
seed_everything(cfg.data.flow.random_state)
eq, sol, dt = make_trajectories(cfg, verbose=True)
print(f"\nFull trajectory shape: {sol['values'].shape}")
print(f"Time step dt = {dt:.4f}")


Full trajectory shape: (32, 1200, 3)
Time step dt = 0.0150


In [11]:
result = postprocess_data(cfg, sol["values"])
values = result.values
mu = result.mu
sigma = result.sigma
noise_scale_factor = result.noise_scale_factor

# Store on cfg for W&B logging
cfg.data.postprocessing.mu = float(mu)
cfg.data.postprocessing.sigma = float(sigma)
cfg.data.postprocessing.noise_scale_factor = float(noise_scale_factor)

print(f"Postprocessed shape: {values.shape}")
print(f"Normalization: mu={mu:.4f}, sigma={sigma:.4f}")
print(f"Noise scale factor: {noise_scale_factor:.4f}")

Postprocessed shape: (32, 1200, 3)
Normalization: mu=0.0000, sigma=1.0000
Noise scale factor: 16.1713


In [12]:
train_dl, val_dl, test_dl, trajs = create_dataloaders(cfg, values, verbose=True)
n_obs = trajs['train_trajs'].sequence.shape[-1]
print(f"\nn_obs = {n_obs}")

# Verify batch shape
sample_batch = next(iter(train_dl))
if isinstance(sample_batch, (list, tuple)):
    sample_batch = sample_batch[0]
print(f"Batch shape: {sample_batch.shape}  (B, T, D_obs)")

Sequence Indices:   0%|          | 0/1058 [00:00<?, ?it/s]

Train dataset shape: torch.Size([23276, 100, 43])
Validation dataset shape: torch.Size([7406, 100, 43])
Test dataset shape: torch.Size([3174, 100, 43])
Train trajectories dataset shape: torch.Size([22, 1158, 43])
Validation trajectories dataset shape: torch.Size([7, 1158, 43])
Test trajectories dataset shape: torch.Size([3, 1158, 43])

n_obs = 43
Batch shape: torch.Size([16, 100, 43])  (B, T, D_obs)


## 6. Build Model

Instantiate the Jacobian MLP from config, then create `LitLatentJacobianODE`
with the pre-trained encoder adapter. The adapter provides `encode()` / `decode()` /
`context_margin` / `n_latent` — everything `LitLatentJacobianODE` needs.

In [13]:
seed_everything(cfg.data.flow.random_state + cfg.training.run_number + 1)

# Instantiate Jacobian MLP from config
jac_model = instantiate(cfg.model.params)

# Build LitLatentJacobianODE with the pre-trained encoder adapter
lit_model = instantiate(
    cfg.training.lightning,
    model=jac_model,
    encoder=adapter,
    dt=dt,
    save_dir=SAVE_DIR,
    mu=float(mu),
    sigma=float(sigma),
    noise_scale_factor=float(noise_scale_factor),
    prediction_steps=PREDICTION_STEPS,
)

# True Lyapunov exponents for logging (Lorenz: sigma=10, rho=28, beta=8/3)
lit_model.true_lyapunov_exponents = torch.tensor([0.91, 0.0, -14.57])

# Parameter summary
encoder_params = sum(p.numel() for p in adapter.parameters())
jac_params = sum(p.numel() for p in jac_model.parameters())
trainable = sum(p.numel() for p in lit_model.parameters() if p.requires_grad)
print(f"Encoder:          {encoder_params:,} params {'(frozen)' if FREEZE_ENCODER else '(trainable)'}")
print(f"Jacobian MLP:     {jac_params:,} params")
print(f"Total trainable:  {trainable:,}")

Encoder:          773,045 params (frozen)
Jacobian MLP:     448,356 params
Total trainable:  448,356


## 7. Train

In [14]:
# Clean up any stale W&B run
try:
    wandb.finish(quiet=True)
except Exception:
    pass

In [15]:
# Auto-generate project and run name
data_cls = cfg.data.flow._target_.split('.')[-1]
if WANDB_PROJECT is None:
    WANDB_PROJECT = f"{data_cls}__PretrainedEncoderJacODE"

freeze_tag = "frozen" if FREEZE_ENCODER else "unfrozen"
encoder_type = type(adapter.encoder).__name__
name = (
    f"{data_cls}__{encoder_type}__n{N_LATENT}__{freeze_tag}"
    f"__lc{LOOP_CLOSURE_WEIGHT}__pred{PREDICTION_STEPS}"
)

print(f"W&B project: {WANDB_PROJECT}")
print(f"Run name:    {name}")

W&B project: Lorenz__PretrainedEncoderJacODE
Run name:    Lorenz__SSMSequenceEncoder__n10__frozen__lc0.001__pred30


In [16]:
train_model(
    cfg=cfg,
    lit_model=lit_model,
    train_dataloaders=train_dl,
    val_dataloaders=val_dl,
    name=name,
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
)

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: Currently logged in as: adamjeisen (JacobianODE) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/home/eisenaj/code/JacobianODE/.venv/lib64/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/eisenaj/code/JacobianODE/.venv/lib64/python3.1 ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type                     | Params | Mode  | FLOPs
-----------------------------------------------------------------------
0 | model     | MLP                      | 448 K  | train | 0    
1 | criterion | MSELoss                  | 0      | train | 0    
2 | encoder   | Pretr

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/eisenaj/code/JacobianODE/.venv/lib64/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/eisenaj/code/JacobianODE/.venv/lib64/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:534: Found 31 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Training: |          | 0/? [00:00<?, ?it/s]

...


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/eisenaj/code/JacobianODE/.venv/lib64/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x1514f8ebfec0>> (for post_run_cell), with arguments args (<ExecutionResult object at 15151149c140, execution_count=16 error_before_exec=None error_in_exec=1 info=<ExecutionInfo object at 15151149c0e0, raw_cell="train_model(
    cfg=cfg,
    lit_model=lit_model,.." transformed_cell="train_model(
    cfg=cfg,
    lit_model=lit_model,.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22656e676167696e672d6e6f6465227d/home/eisenaj/code/JacobianODE/_jupyter/Pretrained%20Encoder%20%2B%20JacobianODE%20%28Lorenz%20-%20single%20run%29.ipynb#Y210sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

## 8. Evaluate the Trained Model

### 8a. Observation-Space Prediction Quality

In [ ]:
# Evaluate on GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lit_model = lit_model.to(device)
lit_model.eval()

val_losses = []
all_z = []
all_obs = []

with torch.no_grad():
    for batch in val_dl:
        if isinstance(batch, (list, tuple)):
            batch = batch[0]
        batch = batch.to(device)

        result = lit_model.trajectory_model_step(
            batch,
            alpha_teacher_forcing=0,
            obs_noise_scale=0,
        )
        val_losses.append(result['loss'].item())

        z = lit_model.encode_trajectory(batch)
        all_z.append(z.cpu())  # move back to cpu to save memory
        all_obs.append(batch.cpu())

print(f"Validation trajectory loss (no teacher forcing): {np.mean(val_losses):.6f}")

z_all = torch.cat(all_z, dim=0)
obs_all = torch.cat(all_obs, dim=0)

NameError: name 'lit_model' is not defined

### 8b. Prediction Visualization

In [ ]:
lit_model.eval()
test_batch = next(iter(test_dl))
if isinstance(test_batch, (list, tuple)):
    test_batch = test_batch[0]

with torch.no_grad():
    z_test = lit_model.encode_trajectory(test_batch)
    window_len = TRAJ_INIT_STEPS + PREDICTION_STEPS
    z_window = z_test[:, :window_len, :]

    from JacobianODE.jacobians.jacobianODE import JacobianODEint
    jac_odeint = JacobianODEint(lit_model.compute_jacobians, dt)
    z_pred = jac_odeint.generate_dynamics(
        z_window,
        alpha_teacher_forcing=0,
        fast_mode=True,
        traj_init_steps=TRAJ_INIT_STEPS,
        interp_pts=INTERP_PTS,
        inner_N=INNER_N,
        inner_path='line',
    )

    z_pred_crop = z_pred[:, TRAJ_INIT_STEPS:, :]
    z_true_crop = z_window[:, TRAJ_INIT_STEPS:, :]
    decoded_pred = lit_model.decode_trajectory(z_pred_crop)
    decoded_true = lit_model.decode_trajectory(z_true_crop)

n_show = min(4, decoded_pred.shape[0])
fig, axes = plt.subplots(n_show, 1, figsize=(12, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i in range(n_show):
    pred_signal = decoded_pred[i].reshape(-1).numpy()
    true_signal = decoded_true[i].reshape(-1).numpy()
    axes[i].plot(true_signal, label="True (decoded from true z)", alpha=0.8)
    axes[i].plot(pred_signal, '--', label="Predicted (decoded from JacODE z)", alpha=0.8)
    axes[i].set_ylabel("x(t)")
    axes[i].set_title(f"Batch element {i}")
    if i == 0:
        axes[i].legend()

axes[-1].set_xlabel("Time index")
fig.suptitle(f"Obs-Space Predictions ({PREDICTION_STEPS} latent steps forward)", y=1.01)
plt.tight_layout()
plt.show()

### 8c. Latent Space Visualization (PCA)

In [ ]:
from sklearn.decomposition import PCA

z_flat = z_all.reshape(-1, z_all.shape[-1]).numpy()
pca = PCA(n_components=3)
z_pca = pca.fit_transform(z_flat)

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(131, projection='3d')
n_plot = min(5000, len(z_pca))
idx = np.random.choice(len(z_pca), n_plot, replace=False)
ax1.scatter(z_pca[idx, 0], z_pca[idx, 1], z_pca[idx, 2], s=0.5, alpha=0.3)
ax1.set_title("Latent Space (PCA)")
ax1.set_xlabel("PC1"); ax1.set_ylabel("PC2"); ax1.set_zlabel("PC3")

ax2 = fig.add_subplot(132)
ax2.scatter(z_pca[idx, 0], z_pca[idx, 1], s=0.5, alpha=0.3)
ax2.set_xlabel("PC1"); ax2.set_ylabel("PC2")
ax2.set_title("Latent PC1 vs PC2")

ax3 = fig.add_subplot(133)
ax3.bar(range(1, 4), pca.explained_variance_ratio_)
ax3.set_xlabel("Component"); ax3.set_ylabel("Variance Explained")
ax3.set_title(f"PCA Variance ({pca.explained_variance_ratio_.sum():.1%} total)")

plt.tight_layout()
plt.show()

### 8d. Lyapunov Exponent Comparison

In [ ]:
TRUE_LYAPUNOV = [0.91, 0.0, -14.57]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lit_model_dev = lit_model.to(device)
lit_model_dev.eval()

with torch.no_grad():
    z_long = z_all[:5].reshape(-1, z_all.shape[-1]).unsqueeze(0).to(device)
    jacs_long = lit_model_dev.compute_jacobians(z_long)[0]
    pred_lyap = LitLatentJacobianODE.compute_lyapunov_exponents(jacs_long, dt)

pred_lyap = pred_lyap.cpu()
lit_model = lit_model.cpu()

print("Predicted Lyapunov exponents (latent space):")
for i, le in enumerate(pred_lyap):
    print(f"  lambda_{i+1} = {le.item():+.4f}")

print(f"\nTrue Lorenz Lyapunov exponents:")
for i, le in enumerate(TRUE_LYAPUNOV):
    print(f"  lambda_{i+1} = {le:+.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
pred_np = pred_lyap.numpy()
ax.bar(np.arange(len(pred_np)) - 0.15, pred_np, width=0.3, label="Predicted (latent)", alpha=0.8)
ax.bar(np.arange(len(TRUE_LYAPUNOV)) + 0.15, TRUE_LYAPUNOV, width=0.3, label="True (Lorenz)", alpha=0.8)
ax.axhline(y=0, color='k', linestyle='--', lw=0.5)
ax.set_xlabel("Exponent index")
ax.set_ylabel("Lyapunov exponent")
ax.set_title("Lyapunov Spectrum: Predicted vs True")
ax.legend()
plt.tight_layout()
plt.show()

### 8e. Latent Utilization & S_dim

In [ ]:
# Latent utilization (entropy-based, [0, 1])
z_var = z_all.reshape(-1, N_LATENT).var(dim=0)
p = z_var / z_var.sum()
utilization = -(p * p.log()).sum() / np.log(N_LATENT)
print(f"Latent utilization: {utilization.item():.4f} (1.0 = all dims equally used)")

# S_dim (if full state is available)
full_state = sol["values"][:5].reshape(-1, sol["values"].shape[-1])
latent_flat = z_all[:5].reshape(-1, N_LATENT).numpy()

vars_true = compute_variances(full_state, normalize=True)
vars_latent = compute_variances(latent_flat, normalize=True)

max_len = max(len(vars_true), len(vars_latent))
vars_true_padded = np.pad(vars_true, (0, max_len - len(vars_true)))
vars_latent_padded = np.pad(vars_latent, (0, max_len - len(vars_latent)))

s_dim = compute_s_dim(vars_true_padded, vars_latent_padded)
print(f"S_dim = {s_dim:.4f} (1.0 = perfect variance alignment)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Variance per latent dimension
axes[0].bar(range(1, N_LATENT + 1), z_var.numpy())
axes[0].set_xlabel("Latent dimension")
axes[0].set_ylabel("Variance")
axes[0].set_title(f"Latent Variance (utilization={utilization.item():.3f})")

# S_dim comparison
axes[1].plot(range(1, len(vars_true) + 1), vars_true, 'o-', label=f"True state ({len(vars_true)}D)")
axes[1].plot(range(1, len(vars_latent) + 1), vars_latent, 's-', label=f"Latent ({len(vars_latent)}D)")
axes[1].set_xlabel("Component (sorted)")
axes[1].set_ylabel("Normalized variance")
axes[1].set_title(f"Variance Spectrum (S_dim = {s_dim:.3f})")
axes[1].legend()

plt.tight_layout()
plt.show()